In [1]:
import brightway2 as bw
from premise import *
import os
import wurst
import time
import copy

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
# bw.projects.set_current('Prospective LCA v5') # set current project
bw.projects.set_current('Prospective LCA - only electricity') # set current project
# bw.projects.set_current('Prospective LCA - only transport') # set current project
# bw.projects.set_current('Prospective LCA - only cement and steel') # set current project
# bw.projects.set_current('Prospective LCA - only technology') # set current project

In [4]:
newLocations = {'BR' : 'Brazil',
                'CA' : 'Canada',
                'PL' : 'Central Europe',
                'CN' : 'China',
                'ET' : 'Eastern Africa',
                'IN' : 'India',
                'ID' : 'Indonesia',
                'JP' : 'Japan',
                'KR' : 'Korea',
                'IR' : 'Middle East',
                'MX' : 'Mexico',
                'EG' : 'Northern Africa',
                'AU' : 'Oceania',
                'GT' : 'Rest of Central America',
                'BW' : 'Rest of Southern Africa',
                'CL' : 'Rest of Southern America',
                'PK' : 'Rest of Southern Asia',
                'RU' : 'Russia',
                'ZA' : 'South Africa',
                'PH' : 'South Eastern Asia',
                'UZ' : 'Central Asia',
                'TR' : 'Turkey',
                'UA' : 'Ukraine',
                'US' : 'United States of America',
                'NG' : 'Western Africa',
                'RER' : 'Western Europe'}

In [5]:
allDBNames = list(bw.databases)

In [6]:
myDBNames = []
for DBName in allDBNames:
    if 'Abhi' in DBName:
        myDBNames.append(DBName)

In [7]:
ecoinventSSP2DBNames = []
for DBName in allDBNames:
    if 'SSP2' in DBName and 'ecoinvent' in DBName:
        ecoinventSSP2DBNames.append(DBName)
ecoinventSSP2DBNames.sort()

In [8]:
imageLocations = {'Global' : 'World',
                'Brazil' : 'BRA',
                'Canada' : 'CAN',
                'Central Europe' : 'CEU',
                'China' : 'CHN',
                'Eastern Africa' : 'EAF',
                'India' : 'INDIA',
                'Indonesia' : 'INDO',
                'Japan' : 'JAP',
                'Korea' : 'KOR',
                'Middle East' : 'ME',
                'Mexico' : 'MEX',
                'Northern Africa' : 'NAF',
                'Oceania' : 'OCE',
                'Rest of Central America' : 'RCAM',
                'Rest of Southern Africa' : 'RSAF',
                'Rest of Southern America' : 'RSAM',
                'Rest of Southern Asia' : 'RSAS',
                'Russia' : 'RUS',
                'South Africa' : 'SAF',
                'South Eastern Asia' : 'SEAS',
                'Central Asia' : 'STAN',
                'Turkey' : 'TUR',
                'Ukraine' : 'UKR',
                'United States of America' : 'USA',
                'Western Africa' : 'WAF',
                'Western Europe' : 'WEU'}

solarLocations = {'Brazil' : 'RoW',
                  'Canada' : 'CA-AB',
                  'Central Europe' : 'RoW',
                  'China' : 'CN-BJ',
                  'Eastern Africa' : 'RoW',
                  'India' : 'RoW',
                  'Indonesia' : 'RoW',
                  'Japan' : 'JP',
                  'Korea' : 'KR',
                  'Middle East' : 'RoW',
                  'Mexico' : 'MX',
                  'Northern Africa' : 'RoW',
                  'Oceania' : 'AU',
                  'Rest of Central America' : 'RoW',
                  'Rest of Southern Africa' : 'RoW',
                  'Rest of Southern America' : 'AR',
                  'Rest of Southern Asia' : 'RoW',
                  'Russia' : 'RoW',
                  'South Africa' : 'RoW',
                  'South Eastern Asia' : 'RoW',
                  'Central Asia' : 'RoW',
                  'Turkey' : 'RoW',
                  'Ukraine' : 'RoW',
                  'United States of America' : 'US-HICC',
                  'Western Africa' : 'RoW',
                  'Western Europe' : 'DE'}

windLocations =  {'Brazil' : 'RoW',
                  'Canada' : 'CA-AB',
                  'Central Europe' : 'BG',
                  'China' : 'CN-AH',
                  'Eastern Africa' : 'RoW',
                  'India' : 'IN-TN',
                  'Indonesia' : 'RoW',
                  'Japan' : 'JP',
                  'Korea' : 'KR',
                  'Middle East' : 'IR',
                  'Mexico' : 'MX',
                  'Northern Africa' : 'RoW',
                  'Oceania' : 'AU',
                  'Rest of Central America' : 'RoW',
                  'Rest of Southern Africa' : 'RoW',
                  'Rest of Southern America' : 'CL',
                  'Rest of Southern Asia' : 'RoW',
                  'Russia' : 'RU',
                  'South Africa' : 'ZA',
                  'South Eastern Asia' : 'RoW',
                  'Central Asia' : 'RoW',
                  'Turkey' : 'TR',
                  'Ukraine' : 'UA',
                  'United States of America' : 'US-ASCC',
                  'Western Africa' : 'RoW',
                  'Western Europe' : 'DE'}

def find_key_by_value(dictionary, value):
    for key, val in dictionary.items():
        if val == value:
            return key
    return None

def find_value_by_key(dictionary, key):
    for k, val in dictionary.items():
        if k == key:
            return val
    return None

In [9]:
# generate all hydrogen inventories
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database(ecoinventSSP2DBName)
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database(mySSP2DBName)
    print('Started ' + mySSP2DBName)

    locations = {'Middle East' : 'ME'}

    for value, location in locations.items():

        ecoinventLocation = find_key_by_value(newLocations, value)
        windLocation = find_value_by_key(windLocations, value)
        solarLocation = find_value_by_key(solarLocations, value)

        if ecoinventLocation:
            newLoc = ecoinventLocation
            
            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and solarLocation == activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and windLocation == activity['location']]
                
        else:
            newLoc = 'GLO'

            solarElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                        and 'electricity, low voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
            
            onshoreWindElecAct = [activity for activity in ecoinventSSP2DB
                        if 'electricity production, wind, >3MW turbine, onshore' in activity['name']
                        and 'electricity, high voltage' in activity['reference product']
                        and 'RoW' in activity['location']]
    
        hydrogenPEMSolarCopy = [activity for activity in mySSP2DB 
                            if 'from PEM electrolysis, from solar electricity' in activity['name']
                            and 'hydrogen, Abhi' in activity['reference product']
                            and newLoc in activity['location']][0]
        elecHydrogenPEMSolarCopys = [exchange for exchange in hydrogenPEMSolarCopy.technosphere()
                                    if 'electricity, low voltage' in exchange['product']]
        for elecHydrogenPEMSolarCopy in elecHydrogenPEMSolarCopys:
            elecHydrogenPEMSolarCopy.input = solarElecAct[0]
            elecHydrogenPEMSolarCopy.save()

        hydrogenPEMOnshoreWindCopy = [activity for activity in mySSP2DB 
                            if 'from PEM electrolysis, from onshore wind electricity' in activity['name']
                            and 'hydrogen, Abhi' in activity['reference product']
                            and newLoc in activity['location']][0]
        elecHydrogenPEMOnshoreWindCopys = [exchange for exchange in hydrogenPEMOnshoreWindCopy.technosphere()
                                    if 'electricity' in exchange['product']]
        for elecHydrogenPEMOnshoreWindCopy in elecHydrogenPEMOnshoreWindCopys:
            elecHydrogenPEMOnshoreWindCopy.input = onshoreWindElecAct[0]
            elecHydrogenPEMOnshoreWindCopy.save()

    print('Finished ', mySSP2DBName)

Started lci-Abhi image SSP2-Base 2020
Finished  lci-Abhi image SSP2-Base 2020
Started lci-Abhi image SSP2-Base 2030
Finished  lci-Abhi image SSP2-Base 2030
Started lci-Abhi image SSP2-Base 2040
Finished  lci-Abhi image SSP2-Base 2040
Started lci-Abhi image SSP2-Base 2050
Finished  lci-Abhi image SSP2-Base 2050
Started lci-Abhi image SSP2-RCP19 2030
Finished  lci-Abhi image SSP2-RCP19 2030
Started lci-Abhi image SSP2-RCP19 2040
Finished  lci-Abhi image SSP2-RCP19 2040
Started lci-Abhi image SSP2-RCP19 2050
Finished  lci-Abhi image SSP2-RCP19 2050
Started lci-Abhi image SSP2-RCP26 2030
Finished  lci-Abhi image SSP2-RCP26 2030
Started lci-Abhi image SSP2-RCP26 2040
Finished  lci-Abhi image SSP2-RCP26 2040
Started lci-Abhi image SSP2-RCP26 2050
Finished  lci-Abhi image SSP2-RCP26 2050
